# 09. Saving & Loading Recurrent Models — Keras (RNN/LSTM)

A trained RNN is only useful if you can persist it. We save & reload both a text classifier and a forecaster, verifying predictions survive the round-trip.

**Two datasets, one concept:**
- **A) IMDB** movie reviews — many-to-one *text classification* (metric: accuracy).
- **B) Jena Climate** weather — sliding-window *temperature forecasting* (metric: MAE).

> Data prep lives in `rnn_data.py`; run `01_sequence_data_prep` once to build the caches.

---

## Setup

In [1]:
import numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import rnn_data as R                      # shared loaders (see rnn_data.py)
keras.utils.set_random_seed(42)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__)

TensorFlow 2.21.0 | Keras 3.14.0


In [2]:
def plot_history(h, metric, title=""):
    """Train (solid) vs validation (dashed): loss on the left, `metric` on the right."""
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h.history["loss"], label="train")
    ax[0].plot(h.history["val_loss"], "--", label="val")
    ax[0].set_title(title + " — loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
    ax[1].plot(h.history[metric], label="train")
    ax[1].plot(h.history["val_" + metric], "--", label="val")
    ax[1].set_title(title + " — " + metric); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

## Part A — IMDB LSTM: full `.keras` + weights-only

In [3]:
Xtr, ytr, Xte, yte = R.load_imdb(); cfg = R.imdb_config()
keras.utils.set_random_seed(42)
clf = keras.Sequential([keras.Input((cfg["maxlen"],)),
                        layers.Embedding(cfg["num_words"], 32), layers.LSTM(32),
                        layers.Dense(1, activation="sigmoid")])
clf.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
clf.fit(Xtr, ytr, epochs=2, batch_size=64, verbose=0)
ref = clf.predict(Xte[:5], verbose=0).ravel()
print("reference preds:", np.round(ref, 5))

# full model (architecture + weights + optimizer)
clf.save("imdb_lstm.keras")
reloaded = keras.models.load_model("imdb_lstm.keras")
print("full-reload identical? ", np.allclose(ref, reloaded.predict(Xte[:5], verbose=0).ravel()))

# weights only (must rebuild the same architecture first)
clf.save_weights("imdb_lstm.weights.h5")
fresh = keras.Sequential([keras.Input((cfg["maxlen"],)),
                          layers.Embedding(cfg["num_words"], 32), layers.LSTM(32),
                          layers.Dense(1, activation="sigmoid")])
fresh.load_weights("imdb_lstm.weights.h5")
print("weights-reload identical?", np.allclose(ref, fresh.predict(Xte[:5], verbose=0).ravel()))

reference preds: [0.13708 0.97588 0.89419 0.10026 0.98932]


full-reload identical?  True


weights-reload identical? True


## Part B — Jena LSTM forecaster: save & reload

In [4]:
train_ds, val_ds, nfeat = R.make_jena_datasets(lookback=48, horizon=1, batch_size=128)
keras.utils.set_random_seed(42)
reg = keras.Sequential([keras.Input((None, nfeat)), layers.LSTM(32), layers.Dense(1)])
reg.compile(optimizer="adam", loss="mse", metrics=["mae"])
reg.fit(train_ds, epochs=2, verbose=0)

# grab one validation batch to test the round-trip
xb = next(iter(val_ds))[0][:5]
ref = reg.predict(xb, verbose=0).ravel()
reg.save("jena_lstm.keras")
reloaded = keras.models.load_model("jena_lstm.keras")
print("Jena preds:", np.round(ref, 4))
print("full-reload identical?", np.allclose(ref, reloaded.predict(xb, verbose=0).ravel()))

Jena preds: [0.3205 0.2606 0.2228 0.1607 0.1231]


full-reload identical? True


## Takeaways
- **`.keras`** (one file: architecture + weights + optimizer) is the everyday format — it round-trips exactly, verified here with `np.allclose`.
- **weights-only** (`.weights.h5`) needs you to rebuild the *identical* architecture first; ideal for checkpoints.
- Recurrent models save exactly like any other Keras model — no special handling. For serving, `model.export()` writes a TensorFlow SavedModel (see the earlier `notebooks_DeepLearning/08` notebook).
- **Next:** `10_text_advanced` and `11_timeseries_advanced` — the dataset-specific concepts.